# BART Ridership Data Audit

This notebook audits the processed ridership pipeline used by the Dash app. The key data decision is:

```text
Hourly OD is the canonical ridership source where available.
Workbook Total Trips / Total Trips OD files are validation benchmarks and fallback sources.
```

The notebook focuses on station-code normalization, source coverage, and differences between hourly OD and workbook totals.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)


def find_project_root(start=None):
    """Find the repo root from either notebooks/ or the repo root itself."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing src/ and data/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REFERENCE = PROJECT_ROOT / "data" / "reference"
PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw"
HOURLY_OD_DIR = RAW / "Hourly_OD"

PROJECT_ROOT


## Reference and Processed Data Inputs

The app should read stable reference tables and processed files. Raw hourly OD CSVs and workbooks are used by `scripts/prepare_data.py` to regenerate processed artifacts and validation outputs.


In [ ]:
station_master_path = REFERENCE / "stations_master.csv"
station_aliases_path = REFERENCE / "station_aliases.csv"
app_summary_path = PROCESSED / "station_ridership_monthly_summary.csv"
hourly_summary_path = PROCESSED / "hourly_od_station_monthly_summary.csv"
validation_path = PROCESSED / "hourly_od_total_trips_validation.csv"
hourly_quality_path = PROCESSED / "hourly_od_completeness_audit.csv"

input_files_df = pd.DataFrame(
    [
        {
            "Artifact": path.name,
            "Role": role,
            "Path": path.relative_to(PROJECT_ROOT).as_posix(),
            "Exists": path.exists(),
            "Size MB": path.stat().st_size / 1_000_000 if path.exists() else None,
        }
        for role, path in [
            ("Station identity reference", station_master_path),
            ("Raw-code alias reference", station_aliases_path),
            ("App monthly station summary", app_summary_path),
            ("Hourly OD monthly station summary", hourly_summary_path),
            ("Hourly OD vs workbook validation", validation_path),
            ("Hourly OD completeness and imputation audit", hourly_quality_path),
        ]
    ]
)
input_files_df


In [ ]:
station_master_df = pd.read_csv(station_master_path, dtype={"station_id": "string"})
station_aliases_df = pd.read_csv(station_aliases_path, dtype={"source_system": "string", "raw_code": "string", "station_id": "string"})
app_summary_df = pd.read_csv(app_summary_path)
hourly_summary_df = pd.read_csv(hourly_summary_path)
valid_ridership_df = pd.read_csv(validation_path)
hourly_quality_df = pd.read_csv(hourly_quality_path)

summary_shapes_df = pd.DataFrame(
    [
        {"Table": "Station master reference", "Rows": len(station_master_df), "Columns": len(station_master_df.columns)},
        {"Table": "Station alias reference", "Rows": len(station_aliases_df), "Columns": len(station_aliases_df.columns)},
        {"Table": "App monthly station summary", "Rows": len(app_summary_df), "Columns": len(app_summary_df.columns)},
        {"Table": "Hourly OD monthly station summary", "Rows": len(hourly_summary_df), "Columns": len(hourly_summary_df.columns)},
        {"Table": "Hourly OD vs workbook validation", "Rows": len(valid_ridership_df), "Columns": len(valid_ridership_df.columns)},
        {"Table": "Hourly OD completeness and imputation audit", "Rows": len(hourly_quality_df), "Columns": len(hourly_quality_df.columns)},
    ]
)
summary_shapes_df


In [ ]:
period_coverage_df = (
    app_summary_df
    .groupby(["Year", "Month", "Period", "Source Type"], as_index=False)
    .agg(
        Stations=("Entry Station", "nunique"),
        Total_Ridership=("Ridership", "sum"),
    )
    .sort_values(["Year", "Month"])
)

period_coverage_display_df = period_coverage_df[
    ["Period", "Source Type", "Stations", "Total_Ridership"]
]

pd.concat([period_coverage_display_df.head(), period_coverage_display_df.tail()])


In [ ]:
source_type_summary_df = (
    app_summary_df
    .groupby("Source Type", as_index=False)
    .agg(
        Rows=("Entry Station", "count"),
        Periods=("Period", "nunique"),
        Total_Ridership=("Ridership", "sum"),
    )
)
source_type_summary_df


## January 2018 Workbook Edge Case

`Ridership_201801.xlsx` does not contain a comparable monthly Total Trips sheet. It contains average weekday/Saturday/Sunday OD sheets, so January 2018 is taken from hourly OD instead.


In [ ]:
jan_2018_workbook = RAW / "ridership_2018" / "Ridership_201801.xlsx"
jan_2018_hourly = HOURLY_OD_DIR / "date-hour-soo-dest-2018.csv"

jan_2018_sheets = pd.ExcelFile(jan_2018_workbook).sheet_names
jan_2018_sheets


In [ ]:
def hourly_month_total(csv_path, year, month, chunksize=500_000):
    """Compute one monthly total without loading the full hourly CSV into memory."""
    period_prefix = f"{year}-{month:02d}"
    total = 0
    for chunk in pd.read_csv(
        csv_path,
        header=None,
        names=["Date", "Hour", "Origin", "Destination", "Exits"],
        usecols=["Date", "Exits"],
        chunksize=chunksize,
    ):
        selected = chunk[chunk["Date"].astype(str).str.startswith(period_prefix)]
        total += pd.to_numeric(selected["Exits"], errors="coerce").fillna(0).sum()
    return int(total)

jan_2018_hourly_total = hourly_month_total(jan_2018_hourly, 2018, 1)
jan_2018_app_total = int(
    app_summary_df.loc[(app_summary_df["Year"] == 2018) & (app_summary_df["Month"] == 1), "Ridership"].sum()
)

pd.DataFrame(
    [
        {"Metric": "January 2018 hourly OD total", "Ridership": jan_2018_hourly_total},
        {"Metric": "January 2018 app summary total", "Ridership": jan_2018_app_total},
    ]
)


## Station Alias Audit

`stations_master.csv` is the clean station dimension table. `station_aliases.csv` is the compact machine-readable crosswalk from raw source codes to official `station_id` values. This section enriches that alias table for human audit purposes without pushing noisy source details back into the master table.


In [ ]:
legacy_app_to_station_id = (
    station_aliases_df[station_aliases_df["source_system"] == "legacy_app_code"]
    .set_index("raw_code")["station_id"]
)

alias_audit_df = station_aliases_df.merge(
    station_master_df[["station_id", "display_name", "opened_date", "closed_date"]],
    on="station_id",
    how="left",
    validate="many_to_one",
)
alias_audit_df["raw_code_matches_station_id"] = (
    alias_audit_df["raw_code"].astype(str).str.lower()
    == alias_audit_df["station_id"].astype(str).str.lower()
)
alias_audit_df = alias_audit_df[
    [
        "source_system",
        "raw_code",
        "station_id",
        "display_name",
        "opened_date",
        "closed_date",
        "raw_code_matches_station_id",
    ]
].sort_values(["source_system", "station_id", "raw_code"]).reset_index(drop=True)

alias_display_df = alias_audit_df[
    ["source_system", "raw_code", "station_id", "display_name", "raw_code_matches_station_id"]
]

alias_display_df


In [ ]:
# If this table is non-empty, an alias points to a station_id missing from stations_master.csv.
alias_target_holes_df = alias_audit_df[alias_audit_df["display_name"].isna()].copy()

alias_target_holes_df


In [ ]:
alias_changes_only_df = alias_audit_df[~alias_audit_df["raw_code_matches_station_id"]].copy()

alias_source_summary_df = (
    alias_audit_df.groupby("source_system", as_index=False)
    .agg(
        alias_rows=("raw_code", "count"),
        covered_station_ids=("station_id", "nunique"),
        aliases_that_change_station_id=("raw_code_matches_station_id", lambda values: int((~values).sum())),
    )
    .sort_values("source_system")
)

alias_source_summary_df


## Raw Hourly OD Code Coverage

This checks raw hourly OD station codes found in each yearly CSV against the `hourly_od` rows in `station_aliases.csv`. Any `Unmapped Raw Codes` value should be investigated before treating hourly OD as canonical for that year.


In [ ]:
def raw_hourly_codes_for_file(csv_path, chunksize=500_000):
    codes = set()
    for chunk in pd.read_csv(
        csv_path,
        header=None,
        names=["Date", "Hour", "Origin", "Destination", "Exits"],
        usecols=["Origin", "Destination"],
        chunksize=chunksize,
    ):
        codes.update(chunk["Origin"].dropna().astype(str).str.strip().str.upper().unique())
        codes.update(chunk["Destination"].dropna().astype(str).str.strip().str.upper().unique())
    return codes


known_hourly_codes = set(
    station_aliases_df.loc[
        station_aliases_df["source_system"] == "hourly_od",
        "raw_code",
    ].astype(str)
)
hourly_code_coverage_rows = []
for csv_path in sorted(HOURLY_OD_DIR.glob("date-hour-soo-dest-*.csv")):
    year = int(csv_path.stem.split("-")[-1])
    raw_codes = raw_hourly_codes_for_file(csv_path)
    hourly_code_coverage_rows.append(
        {
            "Year": year,
            "Raw Code Count": len(raw_codes),
            "Unmapped Raw Codes": sorted(raw_codes - known_hourly_codes),
            "Mapped Raw Codes": sorted(raw_codes & known_hourly_codes),
        }
    )

hourly_code_coverage_df = pd.DataFrame(hourly_code_coverage_rows)
hourly_code_coverage_df[["Year", "Raw Code Count", "Unmapped Raw Codes"]]


## Hourly OD vs Workbook Validation

The validation table compares station-level monthly totals from hourly OD against workbook Total Trips / Total Trips OD. Since hourly OD is canonical where available, workbook values are treated as a benchmark rather than the runtime source of truth.


In [ ]:
validation_audit_df = valid_ridership_df.copy()

if "station_id" not in validation_audit_df.columns:
    validation_audit_df["station_id"] = validation_audit_df["Entry Station"].astype(str).map(legacy_app_to_station_id)

# Normalize display_name defensively so this cell works with both freshly
# regenerated processed files and stale pre-refactor notebook state.
if "display_name" not in validation_audit_df.columns:
    validation_audit_df = validation_audit_df.merge(
        station_master_df[["station_id", "display_name"]],
        on="station_id",
        how="left",
        validate="many_to_one",
    )
else:
    validation_audit_df = validation_audit_df.merge(
        station_master_df[["station_id", "display_name"]].rename(
            columns={"display_name": "display_name_master"}
        ),
        on="station_id",
        how="left",
        validate="many_to_one",
    )
    validation_audit_df["display_name"] = validation_audit_df["display_name"].fillna(
        validation_audit_df["display_name_master"]
    )
    validation_audit_df = validation_audit_df.drop(columns=["display_name_master"])

validation_audit_df["expected_station_id"] = validation_audit_df["Entry Station"].astype(str).map(legacy_app_to_station_id)
validation_audit_df["Station ID Matches Alias"] = validation_audit_df["station_id"] == validation_audit_df["expected_station_id"]
validation_audit_df["Workbook Ridership"] = pd.to_numeric(validation_audit_df["Workbook Ridership"], errors="coerce").fillna(0)
validation_audit_df["Hourly OD Ridership"] = pd.to_numeric(validation_audit_df["Hourly OD Ridership"], errors="coerce").fillna(0)
validation_audit_df["Difference"] = validation_audit_df["Hourly OD Ridership"] - validation_audit_df["Workbook Ridership"]
validation_audit_df["Absolute Difference"] = validation_audit_df["Difference"].abs()
validation_audit_df["Percent Difference vs Workbook"] = (
    validation_audit_df["Difference"]
    / validation_audit_df["Workbook Ridership"].replace(0, pd.NA)
    * 100
)
validation_audit_df["Absolute Percent Difference vs Workbook"] = validation_audit_df["Percent Difference vs Workbook"].abs()
validation_audit_df["Difference Direction"] = validation_audit_df["Difference"].map(
    lambda value: "Hourly higher" if value > 0 else ("Workbook higher" if value < 0 else "Match")
)


def audit_flag(row):
    if pd.isna(row["station_id"]):
        return "Missing processed station_id"
    if pd.isna(row["display_name"]):
        return "Missing processed display_name"
    if not row["Station ID Matches Alias"]:
        return "Processed station_id does not match alias table"
    if row["Workbook Ridership"] == 0 and row["Hourly OD Ridership"] > 0:
        return "Missing workbook benchmark / workbook station code issue"
    if row["Hourly OD Ridership"] == 0 and row["Workbook Ridership"] > 0:
        return "Missing hourly OD benchmark / hourly station alias issue"
    if row["Absolute Difference"] >= 10_000:
        return "Large absolute difference"
    if pd.notna(row["Absolute Percent Difference vs Workbook"]) and row["Absolute Percent Difference vs Workbook"] >= 5:
        return "Large percentage difference"
    if row["Absolute Difference"] > 0:
        return "Small source-definition difference"
    return "Exact match"


validation_audit_df["Audit Flag"] = validation_audit_df.apply(audit_flag, axis=1)
validation_audit_df = validation_audit_df.sort_values(
    ["Audit Flag", "Absolute Difference"],
    ascending=[True, False],
).reset_index(drop=True)

validation_display_df = validation_audit_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Workbook Ridership",
        "Hourly OD Ridership",
        "Difference",
        "Absolute Difference",
        "Percent Difference vs Workbook",
        "Audit Flag",
    ]
]

validation_display_df


In [ ]:
audit_flag_summary_df = (
    validation_audit_df
    .groupby("Audit Flag", as_index=False)
    .agg(
        Rows=("Period", "count"),
        Periods=("Period", "nunique"),
        Stations=("station_id", "nunique"),
        Max_Absolute_Difference=("Absolute Difference", "max"),
        Mean_Absolute_Difference=("Absolute Difference", "mean"),
    )
    .sort_values("Rows", ascending=False)
)
audit_flag_summary_df


In [ ]:
priority_validation_rows_df = validation_audit_df[
    validation_audit_df["Audit Flag"].isin(
        [
            "Missing workbook benchmark / workbook station code issue",
            "Missing hourly OD benchmark / hourly station alias issue",
            "Large absolute difference",
            "Large percentage difference",
        ]
    )
].sort_values("Absolute Difference", ascending=False)

priority_validation_display_df = priority_validation_rows_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Workbook Ridership",
        "Hourly OD Ridership",
        "Difference",
        "Absolute Difference",
        "Percent Difference vs Workbook",
        "Audit Flag",
    ]
]

priority_validation_display_df


In [ ]:
station_validation_summary_df = (
    validation_audit_df
    .groupby(["station_id", "display_name"], dropna=False)
    .agg(
        Periods_Compared=("Period", "nunique"),
        Rows_With_Difference=("Has Difference", "sum"),
        Max_Absolute_Difference=("Absolute Difference", "max"),
        Mean_Absolute_Difference=("Absolute Difference", "mean"),
        Total_Absolute_Difference=("Absolute Difference", "sum"),
        Max_Absolute_Percent_Difference=("Absolute Percent Difference vs Workbook", "max"),
        Missing_Hourly_Rows=("Audit Flag", lambda values: (values == "Missing hourly OD benchmark / hourly station alias issue").sum()),
        Missing_Workbook_Rows=("Audit Flag", lambda values: (values == "Missing workbook benchmark / workbook station code issue").sum()),
        Large_Absolute_Difference_Rows=("Audit Flag", lambda values: (values == "Large absolute difference").sum()),
        Large_Percentage_Difference_Rows=("Audit Flag", lambda values: (values == "Large percentage difference").sum()),
    )
    .reset_index()
    .sort_values(["Missing_Hourly_Rows", "Missing_Workbook_Rows", "Max_Absolute_Difference"], ascending=False)
)

station_validation_display_df = station_validation_summary_df[
    [
        "station_id",
        "display_name",
        "Periods_Compared",
        "Rows_With_Difference",
        "Max_Absolute_Difference",
        "Mean_Absolute_Difference",
        "Total_Absolute_Difference",
        "Max_Absolute_Percent_Difference",
        "Large_Absolute_Difference_Rows",
        "Large_Percentage_Difference_Rows",
    ]
]

station_validation_display_df


## Period-Level Source Differences

Period-level differences reveal months where the two sources diverge broadly, which is different from a single station-code mapping issue.


In [ ]:
period_validation_summary_df = (
    validation_audit_df
    .groupby(["Year", "Month", "Period"], as_index=False)
    .agg(
        Workbook_Total=("Workbook Ridership", "sum"),
        Hourly_OD_Total=("Hourly OD Ridership", "sum"),
        Total_Absolute_Station_Difference=("Absolute Difference", "sum"),
        Max_Station_Difference=("Absolute Difference", "max"),
        Stations_Compared=("station_id", "nunique"),
        Priority_Row_Count=(
            "Audit Flag",
            lambda values: values.isin(
                [
                    "Missing workbook benchmark / workbook station code issue",
                    "Missing hourly OD benchmark / hourly station alias issue",
                    "Large absolute difference",
                    "Large percentage difference",
                ]
            ).sum(),
        ),
    )
)
period_validation_summary_df["Total Difference"] = period_validation_summary_df["Hourly_OD_Total"] - period_validation_summary_df["Workbook_Total"]
period_validation_summary_df["Total Percent Difference vs Workbook"] = (
    period_validation_summary_df["Total Difference"]
    / period_validation_summary_df["Workbook_Total"].replace(0, pd.NA)
    * 100
)
period_validation_summary_df = period_validation_summary_df.sort_values("Max_Station_Difference", ascending=False)

period_validation_display_df = period_validation_summary_df[
    [
        "Period",
        "Workbook_Total",
        "Hourly_OD_Total",
        "Total Difference",
        "Total Percent Difference vs Workbook",
        "Total_Absolute_Station_Difference",
        "Max_Station_Difference",
        "Stations_Compared",
        "Priority_Row_Count",
    ]
]

period_validation_display_df


## Hourly OD Completeness Audit

This table is produced by `scripts/prepare_data.py` as `data/processed/hourly_od_completeness_audit.csv`. It documents missing full days that were imputed before monthly summaries were written and flags major source-definition discrepancies where hourly OD remains canonical but workbook totals are useful context.


In [ ]:
hourly_completeness_audit_df = hourly_quality_df.copy()

hourly_completeness_display_df = hourly_completeness_audit_df[
    [
        "Period",
        "Quality Flag",
        "Calendar Days",
        "Observed Days",
        "Missing Full Days",
        "Missing Full Day List",
        "Min Present Hours",
        "Median Present Hours",
        "Raw Hourly Ridership",
        "Imputed Hourly Ridership",
        "Imputed Ridership Added",
        "Workbook Ridership",
        "Workbook Difference Pct",
    ]
].sort_values(["Quality Flag", "Period"]).reset_index(drop=True)

hourly_completeness_display_df


In [ ]:
hourly_completeness_flag_summary_df = (
    hourly_completeness_audit_df.groupby("Quality Flag", as_index=False)
    .agg(
        periods=("Period", "count"),
        max_abs_workbook_diff_pct=("Workbook Difference Pct", lambda values: values.abs().max()),
        periods_with_missing_full_days=("Missing Full Days", lambda values: (values > 0).sum()),
        imputed_ridership_added=("Imputed Ridership Added", "sum"),
    )
    .sort_values("Quality Flag")
)

hourly_completeness_flag_summary_df


## Processed Source Coverage Audit

The app summary should be hourly-derived for every available hourly OD period. This table checks coverage after mapping current processed `Entry Station` codes through `legacy_app_code` aliases into official `station_id` values.


In [ ]:
def add_station_identity(df):
    result = df.copy()
    if "station_id" not in result.columns:
        result["station_id"] = result["Entry Station"].astype(str).map(legacy_app_to_station_id)
    if "display_name" not in result.columns:
        result = result.merge(
            station_master_df[["station_id", "display_name"]],
            on="station_id",
            how="left",
            validate="many_to_one",
        )
    return result


app_keys = add_station_identity(app_summary_df[["Year", "Month", "Period", "Entry Station", "station_id", "display_name", "Source Type"]])
app_keys = app_keys[["Year", "Month", "Period", "station_id", "display_name", "Source Type"]].copy()
app_keys["In App Summary"] = True

hourly_keys = add_station_identity(hourly_summary_df[["Year", "Month", "Period", "Entry Station", "station_id", "display_name"]])
hourly_keys = hourly_keys[["Year", "Month", "Period", "station_id", "display_name"]].copy()
hourly_keys["In Hourly Summary"] = True

# Only nonzero benchmark rows are used for coverage-hole detection.
# Some workbooks include unopened future stations as explicit zero rows.
workbook_keys = add_station_identity(
    validation_audit_df[
        (validation_audit_df["Workbook Ridership"] > 0)
        | (validation_audit_df["Hourly OD Ridership"] > 0)
    ][["Year", "Month", "Period", "Entry Station", "station_id", "display_name"]]
)
workbook_keys = workbook_keys[["Year", "Month", "Period", "station_id", "display_name"]].copy()
workbook_keys["In Workbook Benchmark"] = True

coverage_audit_df = (
    app_keys
    .merge(hourly_keys, on=["Year", "Month", "Period", "station_id", "display_name"], how="outer")
    .merge(workbook_keys, on=["Year", "Month", "Period", "station_id", "display_name"], how="outer")
)
for column in ["In App Summary", "In Hourly Summary", "In Workbook Benchmark"]:
    coverage_audit_df[column] = coverage_audit_df[column].fillna(False)


def coverage_flag(row):
    if pd.isna(row["station_id"]):
        return "Missing processed station_id"
    if pd.isna(row["display_name"]):
        return "Missing processed display_name"
    if not row["In App Summary"]:
        return "Missing from app summary"
    if not row["In Hourly Summary"]:
        return "Missing from hourly summary"
    if not row["In Workbook Benchmark"]:
        return "No workbook benchmark row"
    return "Covered"


coverage_audit_df["Coverage Flag"] = coverage_audit_df.apply(coverage_flag, axis=1)
coverage_display_df = coverage_audit_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Source Type",
        "In App Summary",
        "In Hourly Summary",
        "In Workbook Benchmark",
        "Coverage Flag",
    ]
]

coverage_display_df.sort_values(["Coverage Flag", "Period", "station_id"])


In [ ]:
coverage_holes_df = coverage_audit_df[coverage_audit_df["Coverage Flag"] != "Covered"].copy()
coverage_holes_display_df = coverage_holes_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Source Type",
        "In App Summary",
        "In Hourly Summary",
        "In Workbook Benchmark",
        "Coverage Flag",
    ]
]

coverage_holes_display_df.sort_values(["Coverage Flag", "Period", "station_id"])


## Audit Checklist

Use this checklist before changing the app's source-of-truth rules or adding Parquet/time-lapse features.

- `stations_master_df["station_id"]` should be unique and non-null.
- `alias_target_holes_df` should be empty.
- `hourly_code_coverage_df["Unmapped Raw Codes"]` should be empty for every year.
- `hourly_completeness_display_df` should explicitly identify missing full days that were imputed and major source-definition discrepancies where hourly OD remains canonical.
- `coverage_holes_df` should contain only expected benchmark gaps, especially Jan 2018 with no comparable workbook Total Trips sheet.
- `priority_validation_rows_df` should be reviewed and categorized before claiming workbook/hourly agreement.
- Periods with large total percent differences should be investigated separately from station-code mapping issues.
